# CNN-SRM — Cross-Database Deepfake Detection Experiment

**Research question:** Does a noise-pattern detector trained on one deepfake dataset generalize to others?

**Architecture (aligned with main notebook `the_modles_fixed_new`):**
- 8 SRM high-pass filters → **raw residuals** (no tanh squashing)
- SE-Residual autoencoder → frozen encoder (16×16×256 latent)
- Classifier head: **3 conv blocks** (Conv-BN-LeakyReLU-Dropout) → GAP → Dense → sigmoid
- Single-phase training: **encoder always frozen** with `training=False`
- Label smoothing 0.05, no brightness augmentation on SRM features

**Per-database pipeline:**
1. Apply SRM filters (8 filters, 5×5) → (256, 256, 24) raw residual maps
2. Train fresh autoencoder on that database's SRM features (Option A — no leakage)
3. Extract encoder; freeze it
4. Train classifier head end-to-end (encoder fixed)
5. Evaluate on **all 5 test sets** (4 individual + ALL combined)

**Experiment matrix:** 5 training runs × 5 test sets = **25 evaluations**

| Train ↓ \ Test → | OpenForensics | CustomWar | CelebDF | CiFake | ALL |
|------------------|---------------|-----------|---------|--------|-----|
| OpenForensics    | ✓ within      | ✗ cross   | ✗ cross | ✗ cross | mix |
| CustomWar        | ✗ cross       | ✓ within  | ✗ cross | ✗ cross | mix |
| CelebDF          | ✗ cross       | ✗ cross   | ✓ within | ✗ cross | mix |
| CiFake           | ✗ cross       | ✗ cross   | ✗ cross  | ✓ within | mix |
| ALL              | mix           | mix       | mix     | mix    | ✓ within |

**Databases:** OpenForensics · CustomWar · CelebDF · CiFake (CiFake uses the 80/20 split from `cifake_split/`)

In [ ]:
# =================== CELL 1: SETUP ===================
import os
import gc
import random
import pickle
import numpy as np
import tensorflow as tf
import mimetypes
from datetime import datetime

os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/lib/nvidia-cuda-toolkit'
tf.config.optimizer.set_jit(False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print('GPU memory growth enabled')
    except RuntimeError as e:
        print(e)

tf.keras.backend.clear_session()
print(f'Setup complete — TensorFlow {tf.__version__}')

In [ ]:
# =================== CELL 2: PATHS & TENSORBOARD CONFIG ===================

GDRIVE_PATH   = os.path.expanduser('~/RealEyes/gdrive')
DATASET_ROOT  = os.path.join(GDRIVE_PATH, 'data_set_split')
DATASETS_DIR  = os.path.expanduser('~/RealEyes/RealEyes/datasets')
CELEBDF_DIR   = os.path.join(DATASETS_DIR, 'celebdf_v2')
CIFAKE_SPLIT_DIR = os.path.join(DATASETS_DIR, 'cifake_split')   # 80/20 split from main notebook

MODEL_NAME            = 'cnn_srm'
EXPERIMENT_MODELS_DIR = os.path.join(GDRIVE_PATH, 'deepfake_image_project', 'models', 'RealEyes_experiment')
MODEL_DIR             = os.path.join(EXPERIMENT_MODELS_DIR, MODEL_NAME)
os.makedirs(MODEL_DIR, exist_ok=True)

TB_LOG_ROOT = os.path.expanduser('~/RealEyes/tensorboard_logs')
os.makedirs(TB_LOG_ROOT, exist_ok=True)


def get_tb_log_dir(train_db_name, suffix='train'):
    ts = datetime.now().strftime('%Y%m%d_%H%M')
    return os.path.join(TB_LOG_ROOT, MODEL_NAME, f'{suffix}_{train_db_name}', ts)


def log_eval_to_tb(train_db_name, test_db_name, metrics: dict, step=0):
    log_dir = os.path.join(
        TB_LOG_ROOT, MODEL_NAME, 'cross_db_eval',
        f'train_{train_db_name}__test_{test_db_name}'
    )
    writer = tf.summary.create_file_writer(log_dir)
    with writer.as_default():
        for name, value in metrics.items():
            tf.summary.scalar(name, float(value), step=step)
    writer.flush()


print('Paths ready')
print(f'  MODEL_DIR        : {MODEL_DIR}')
print(f'  CIFAKE_SPLIT_DIR : {CIFAKE_SPLIT_DIR}')
print(f'  TB_LOG_ROOT      : {TB_LOG_ROOT}')
print()
print('View TensorBoard:')
print(f'  [server ] tensorboard --logdir {TB_LOG_ROOT} --port 6006 --bind_all')
print( '  [local  ] ssh -L 6006:localhost:6006 <user>@<server_ip>')
print( '  [browser] http://localhost:6006')

In [ ]:
# =================== CELL 3: DATABASE DEFINITIONS ===================
# CiFake now uses the 80/20 split (cifake_split/{train,val,test}) created
# by the main notebook's Section 3.3 — so it has a real held-out test set.

DATABASES = {}


def _try_add(name, paths):
    missing = [k for k, v in paths.items() if not os.path.isdir(v)]
    if missing:
        print(f'  {name}: missing splits {missing} — skipped')
        return
    DATABASES[name] = paths
    print(f'  {name}')


print('Scanning databases...')

_try_add('OpenForensics', {
    'train': os.path.join(DATASETS_DIR, 'OpenForensicsV1/Dataset/Train'),
    'val':   os.path.join(DATASETS_DIR, 'OpenForensicsV1/Dataset/Validation'),
    'test':  os.path.join(DATASETS_DIR, 'OpenForensicsV1/Dataset/Test'),
})
_try_add('CustomWar', {
    'train': os.path.join(DATASET_ROOT, 'train'),
    'val':   os.path.join(DATASET_ROOT, 'val'),
    'test':  os.path.join(DATASET_ROOT, 'test'),
})
_try_add('CelebDF', {
    'train': os.path.join(CELEBDF_DIR, 'train'),
    'val':   os.path.join(CELEBDF_DIR, 'val'),
    'test':  os.path.join(CELEBDF_DIR, 'test'),
})
_try_add('CiFake', {
    'train': os.path.join(CIFAKE_SPLIT_DIR, 'train'),
    'val':   os.path.join(CIFAKE_SPLIT_DIR, 'val'),
    'test':  os.path.join(CIFAKE_SPLIT_DIR, 'test'),
})

print(f'\nActive databases: {list(DATABASES.keys())}')

In [ ]:
# =================== CELL 4: DATA LOADING HELPERS ===================
# load_split() supports both real DB names and the special 'ALL' name
# (concatenation of every database's split).

def load_dataset_images(dataset_path, max_images=None):
    valid_ext = {'.jpg', '.jpeg', '.png', '.gif', '.bmp'}
    image_paths, labels, skipped = [], [], 0

    for folder in sorted(os.listdir(dataset_path)):
        fpath = os.path.join(dataset_path, folder)
        if not os.path.isdir(fpath):
            continue
        fu = folder.upper()
        if fu == 'FAKE':
            label = 1
        elif fu == 'REAL':
            label = 0
        else:
            print(f'  Unknown folder "{folder}" — skipped')
            continue

        collected = []
        for root, _, files in os.walk(fpath):
            for fname in files:
                if os.path.splitext(fname)[1].lower() not in valid_ext:
                    skipped += 1
                    continue
                collected.append(os.path.join(root, fname))
        if max_images:
            collected = collected[:max_images]
        image_paths.extend(collected)
        labels.extend([label] * len(collected))

    if skipped:
        print(f'  {skipped} non-image files skipped')
    return np.array(image_paths), np.array(labels)


def load_split(target_name, split='train'):
    """Load (paths, labels) for a target.

    target_name: a real DB key in DATABASES, OR the literal string 'ALL'
                 which concatenates every database's split.
    """
    if target_name == 'ALL':
        all_paths, all_labels = [], []
        for db_name in DATABASES:
            paths, labels = load_dataset_images(DATABASES[db_name][split])
            all_paths.extend(paths)
            all_labels.extend(labels)
        paths  = np.array(all_paths)
        labels = np.array(all_labels)
    else:
        paths, labels = load_dataset_images(DATABASES[target_name][split])

    n_real = int(np.sum(labels == 0))
    n_fake = int(np.sum(labels == 1))
    print(f'    {target_name}/{split}: {len(paths):,} images  (REAL={n_real:,}, FAKE={n_fake:,})')
    return paths, labels


def compute_class_weights(labels):
    from sklearn.utils.class_weight import compute_class_weight
    classes = np.unique(labels)
    weights = compute_class_weight('balanced', classes=classes, y=labels)
    return {int(c): float(w) for c, w in zip(classes, weights)}


print('Data loading helpers ready (load_split supports ALL)')

In [ ]:
# =================== CELL 5: SRM FILTER BANK ===================
# 8 handcrafted noise-sensitive filters (5 × 5×5 + 3 padded 3×3).
# Identical to the main notebook.

filters_5x5 = [
    [[0, 0, -1, 0, 0], [0, -1, 2, -1, 0], [-1, 2, 4, 2, -1], [0, -1, 2, -1, 0], [0, 0, -1, 0, 0]],
    [[-1, 2, -2, 2, -1], [2, -6, 8, -6, 2], [-2, 8, -12, 8, -2], [2, -6, 8, -6, 2], [-1, 2, -2, 2, -1]],
    [[2, -1, 0, -1, 2], [-1, -2, 3, -2, -1], [0, 3, 0, 3, 0], [-1, -2, 3, -2, -1], [2, -1, 0, -1, 2]],
    [[0, 0, 0, 0, 0], [1, -2, 1, -2, 1], [0, 0, 0, 0, 0], [-1, 2, -1, 2, -1], [0, 0, 0, 0, 0]],
    [[1, -4, 6, -4, 1], [-4, 16, -24, 16, -4], [6, -24, 36, -24, 6], [-4, 16, -24, 16, -4], [1, -4, 6, -4, 1]],
]
filters_3x3_raw = [
    [[0, -1, 0], [-1, 4, -1], [0, -1, 0]],
    [[-1, 2, -1], [2, -4, 2], [-1, 2, -1]],
    [[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]],
]


def _pad_3x3_to_5x5(kernels):
    return [np.pad(k, ((1, 1), (1, 1)), mode='constant', constant_values=0) for k in kernels]


all_filters = np.array(filters_5x5 + _pad_3x3_to_5x5(filters_3x3_raw), dtype=np.float32)
srm_filters_tf = tf.constant(
    np.transpose(all_filters[:, :, :, np.newaxis], (1, 2, 3, 0)),
    dtype=tf.float32
)

print(f'SRM filter bank ready: {len(all_filters)} filters, TF kernel {srm_filters_tf.shape}')

In [ ]:
# =================== CELL 6: SRM FILTER FUNCTION & DATASET PIPELINE ===================
# Aligned with main notebook §5:
#   - apply_srm_filters_tf() returns RAW residuals (no tanh squashing)
#   - SRM augmentation = flips only (brightness on residuals has no clean meaning)

AUTOTUNE         = tf.data.AUTOTUNE
SRM_IMG_SIZE     = (256, 256)
SRM_OUTPUT_SHAPE = (256, 256, 24)


def apply_srm_filters_tf(image):
    """
    Apply 8 SRM filters per RGB channel → (256, 256, 24) RAW residuals.
    Accepts (H, W, 3) or (B, H, W, 3). Input pixels in [0, 1].
    NO tanh — keeps the high-frequency signal intact.
    """
    squeeze = False
    if len(image.shape) == 3:
        image = image[tf.newaxis, ...]
        squeeze = True

    image = tf.image.resize(image, list(SRM_IMG_SIZE))
    image = tf.cast(image, tf.float32)
    channels = tf.split(image, num_or_size_splits=3, axis=-1)

    feature_maps = []
    for ch in channels:
        fm = tf.nn.conv2d(ch, srm_filters_tf, strides=1, padding='SAME')
        feature_maps.append(fm)               # raw residuals — no nonlinearity

    result = tf.concat(feature_maps, axis=-1)
    if squeeze:
        result = result[0]
    return result


def _srm_augment(srm, label):
    # Flips only. Brightness/contrast on residuals doesn't mean anything sensible.
    srm = tf.image.random_flip_left_right(srm)
    srm = tf.image.random_flip_up_down(srm)
    return srm, label


def create_srm_dataset(image_paths, labels, batch_size=64, shuffle=False, augment=False):
    def process(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, list(SRM_IMG_SIZE))
        img = tf.cast(img, tf.float32) / 255.0
        srm = apply_srm_filters_tf(img)
        srm = tf.ensure_shape(srm, list(SRM_OUTPUT_SHAPE))
        return srm, tf.cast(label, tf.float32)

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    if shuffle:
        ds = ds.shuffle(min(len(image_paths), 10000), reshuffle_each_iteration=True)
    ds = ds.map(process, num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(_srm_augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds


print('SRM filter function (raw residuals) and dataset pipeline ready')
print(f'   Output shape per image: {SRM_OUTPUT_SHAPE}')

In [ ]:
# =================== CELL 7: AUTOENCODER MODEL BUILDER ===================
# SE-Residual autoencoder — identical to main notebook §6.
# Decoder activation = linear (SRM features are no longer in [0,1]).

from tensorflow.keras import layers, Model

REG = tf.keras.regularizers.l2(1e-4)


def _conv_bn_lrelu(x, filters, kernel=3, name=''):
    x = layers.Conv2D(filters, kernel, padding='same', use_bias=False,
                      kernel_regularizer=REG, name=f'{name}_conv')(x)
    x = layers.BatchNormalization(momentum=0.9, epsilon=1e-5, name=f'{name}_bn')(x)
    x = layers.LeakyReLU(negative_slope=0.1, name=f'{name}_act')(x)
    return x


def _se_block(x, ratio=16, name=''):
    ch = x.shape[-1]
    r  = max(1, ch // ratio)
    se = layers.GlobalAveragePooling2D(keepdims=True, name=f'{name}_gap')(x)
    se = layers.Dense(r,  activation='relu',    use_bias=False, name=f'{name}_fc1')(se)
    se = layers.Dense(ch, activation='sigmoid', use_bias=False, name=f'{name}_fc2')(se)
    return layers.Multiply(name=f'{name}_scale')([x, se])


def _res_se_block(x, filters, name=''):
    shortcut = x
    x = _conv_bn_lrelu(x, filters, name=f'{name}_a')
    x = _conv_bn_lrelu(x, filters, name=f'{name}_b')
    x = _se_block(x, ratio=16, name=f'{name}_se')
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, padding='same', use_bias=False,
                                 kernel_regularizer=REG, name=f'{name}_proj')(shortcut)
    return layers.Add(name=f'{name}_add')([x, shortcut])


def build_srm_autoencoder():
    """Build SE-Residual SRM autoencoder. Returns (autoencoder, encoder)."""
    inp = layers.Input(shape=(256, 256, 24), name='srm_input')

    # Encoder 256→128→64→32→16
    x = _conv_bn_lrelu(inp, 32, name='enc_s1');  x = _res_se_block(x, 32,  name='enc_res1')
    x = layers.MaxPooling2D((2, 2), name='enc_pool1')(x)
    x = _conv_bn_lrelu(x, 64, name='enc_s2');    x = _res_se_block(x, 64,  name='enc_res2')
    x = layers.MaxPooling2D((2, 2), name='enc_pool2')(x)
    x = _conv_bn_lrelu(x, 128, name='enc_s3');   x = _res_se_block(x, 128, name='enc_res3')
    x = layers.MaxPooling2D((2, 2), name='enc_pool3')(x)
    x = _conv_bn_lrelu(x, 256, name='enc_s4');   x = _res_se_block(x, 256, name='enc_res4')
    encoded = layers.MaxPooling2D((2, 2), name='encoded_latent')(x)   # 16×16×256

    # Decoder 16→32→64→128→256
    x = _conv_bn_lrelu(encoded, 256, name='dec_s1'); x = layers.UpSampling2D((2, 2), name='dec_up1')(x)
    x = _conv_bn_lrelu(x, 128, name='dec_s2');       x = layers.UpSampling2D((2, 2), name='dec_up2')(x)
    x = _conv_bn_lrelu(x, 64, name='dec_s3');        x = layers.UpSampling2D((2, 2), name='dec_up3')(x)
    x = _conv_bn_lrelu(x, 32, name='dec_s4');        x = layers.UpSampling2D((2, 2), name='dec_up4')(x)
    decoded = layers.Conv2D(24, 3, activation='linear', padding='same',
                            name='decoder_output')(x)               # linear — raw residuals

    autoencoder = Model(inp, decoded, name='srm_autoencoder_24ch')
    encoder     = Model(inp, encoded, name='srm_encoder_24ch')

    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss='mse'
    )
    return autoencoder, encoder


print('SRM autoencoder builder ready (linear decoder)')
_ae, _enc = build_srm_autoencoder()
print(f'  Autoencoder output: {_ae.output_shape}')
print(f'  Encoder output:     {_enc.output_shape}')
print(f'  Autoencoder params: {_ae.count_params():,}')
del _ae, _enc
tf.keras.backend.clear_session()

In [ ]:
# =================== CELL 8: CNN-SRM CLASSIFIER BUILDER ===================
# Aligned with main notebook §7. Three conv blocks AFTER the encoder, BEFORE GAP.
# Encoder is called with training=False so its BN moving averages don't drift
# during the classifier's training pass.

def build_cnn_srm(encoder):
    """Build full CNN-SRM model: SRM (256,256,24) → frozen encoder → 3 conv blocks → GAP → Dense → sigmoid."""
    L2 = tf.keras.regularizers.l2(1e-4)

    cls_input = layers.Input(shape=encoder.output_shape[1:], name='latent_input')
    y = cls_input

    # 3 conv blocks BEFORE GAP — gives the head spatial capacity
    for i, f in enumerate([256, 256, 128], start=1):
        y = layers.Conv2D(f, 3, padding='same', kernel_regularizer=L2,
                          name=f'head_conv{i}')(y)
        y = layers.BatchNormalization(name=f'head_bn{i}')(y)
        y = layers.LeakyReLU(0.1, name=f'head_lrelu{i}')(y)
        y = layers.Dropout(0.2, name=f'head_drop{i}')(y)

    y = layers.GlobalAveragePooling2D(name='head_gap')(y)
    y = layers.Dense(128, kernel_regularizer=L2, name='head_dense1')(y)
    y = layers.BatchNormalization(name='head_bn_d1')(y)
    y = layers.Activation('relu', name='head_act_d1')(y)
    y = layers.Dropout(0.4, name='head_drop_d1')(y)

    output = layers.Dense(1, activation='sigmoid', name='prob_fake')(y)
    cls_head = Model(cls_input, output, name='latent_classifier_head')

    full_input = layers.Input(shape=(256, 256, 24), name='srm_input')
    latent     = encoder(full_input, training=False)        # frozen encoder, BN inference mode
    pred       = cls_head(latent)
    model      = Model(full_input, pred, name='SRM_Encoder_Classifier')
    return model


print('CNN-SRM classifier builder ready (deeper head, encoder always frozen)')

In [ ]:
# =================== CELL 9: EVALUATION HELPERS ===================

from sklearn.metrics import classification_report, roc_auc_score, accuracy_score


def evaluate_model(model, test_ds):
    y_true, y_prob = [], []
    for x_batch, y_batch in test_ds:
        preds = model.predict(x_batch, verbose=0).flatten()
        y_true.extend(y_batch.numpy())
        y_prob.extend(preds)

    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob)
    y_pred = (y_prob > 0.5).astype(int)

    report = classification_report(
        y_true, y_pred, target_names=['REAL', 'FAKE'], output_dict=True, digits=4
    )
    metrics = {
        'accuracy':       float(accuracy_score(y_true, y_pred)),
        'roc_auc':        float(roc_auc_score(y_true, y_prob)),
        'f1_fake':        float(report['FAKE']['f1-score']),
        'f1_real':        float(report['REAL']['f1-score']),
        'precision_fake': float(report['FAKE']['precision']),
        'recall_fake':    float(report['FAKE']['recall']),
    }
    return metrics, report


def print_eval_report(train_db, test_db, metrics, report):
    if train_db == test_db:
        tag = '  [WITHIN]   '
    elif train_db == 'ALL' or test_db == 'ALL':
        tag = '  [ALL-MIX]  '
    else:
        tag = '  [CROSS]    '
    print(f'\n{tag}Trained on {train_db:<15} → Tested on {test_db}')
    sep = '  ' + '-' * 58
    print(sep)
    for cls in ['REAL', 'FAKE']:
        r = report[cls]
        print(f'  {cls:<6}  P={r["precision"]:.4f}  R={r["recall"]:.4f}  F1={r["f1-score"]:.4f}  n={int(r["support"]):,}')
    print(sep)
    print(f'  Accuracy={metrics["accuracy"]:.4f}   ROC-AUC={metrics["roc_auc"]:.4f}')


print('Evaluation helpers ready')

In [ ]:
# =================== CELL 10: MAIN EXPERIMENT LOOP (5×5) ===================
#
# Five training runs (4 single-DB + 1 ALL combined), each evaluated on the
# 4 individual test sets PLUS the ALL combined test set.
#
# For each training target:
#   Step A — Train fresh autoencoder on that target's SRM features (no leakage)
#   Step B — Build CNN-SRM with frozen encoder + deeper head, train end-to-end
#            (single phase, encoder stays frozen the whole time)
#   Step C — Evaluate on all 5 test sets (4 individual + ALL)
# ─────────────────────────────────────────────────────────────────────────

BATCH_SIZE     = 64
TRAIN_TARGETS  = list(DATABASES.keys()) + ['ALL']
TEST_TARGETS   = list(DATABASES.keys()) + ['ALL']
all_results    = {}   # all_results[train_target][test_target] = metrics_dict

for train_target in TRAIN_TARGETS:
    print(f'\n{"="*70}')
    print(f'  TRAINING CNN-SRM ON: {train_target}')
    print(f'{"="*70}')

    # ── STEP 0: load training data ────────────────────────────────────────
    print('\nLoading training data...')
    train_paths, train_lbls = load_split(train_target, 'train')
    val_paths,   val_lbls   = load_split(train_target, 'val')
    class_weights           = compute_class_weights(train_lbls)
    print(f'  Class weights: {class_weights}')

    print('\nBuilding SRM datasets...')
    train_srm_ds = create_srm_dataset(
        train_paths, train_lbls, batch_size=BATCH_SIZE, shuffle=True, augment=True)
    val_srm_ds   = create_srm_dataset(
        val_paths, val_lbls, batch_size=BATCH_SIZE, shuffle=False)

    # ── STEP A: fresh autoencoder ─────────────────────────────────────────
    print(f'\n[STEP A] Training SRM autoencoder on {train_target}...')
    gc.collect(); tf.keras.backend.clear_session()
    autoencoder, encoder = build_srm_autoencoder()

    ae_log       = get_tb_log_dir(train_target, suffix='autoencoder')
    encoder_path = os.path.join(MODEL_DIR, f'encoder_{train_target}.keras')
    ae_path      = os.path.join(MODEL_DIR, f'autoencoder_{train_target}.keras')

    ae_train_ds = train_srm_ds.map(lambda x, y: (x, x))
    ae_val_ds   = val_srm_ds.map  (lambda x, y: (x, x))

    ae_callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            ae_path, monitor='val_loss', save_best_only=True, verbose=1),
        tf.keras.callbacks.TensorBoard(
            log_dir=ae_log, histogram_freq=0, update_freq='epoch'),
    ]
    autoencoder.fit(
        ae_train_ds, validation_data=ae_val_ds,
        epochs=30, callbacks=ae_callbacks, verbose=1
    )
    encoder.save(encoder_path)
    print(f'  Encoder saved → {encoder_path}')

    # ── STEP B: train classifier (encoder frozen, single phase) ──────────
    print(f'\n[STEP B] Training CNN-SRM classifier (encoder FROZEN, deeper head)...')
    encoder.trainable = False
    cnn_srm = build_cnn_srm(encoder)
    cnn_srm.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.05),
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )

    tb_log    = get_tb_log_dir(train_target, suffix='train')
    best_path = os.path.join(MODEL_DIR, f'trained_on_{train_target}.keras')

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc', mode='max', patience=12,
            restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_auc', mode='max', factor=0.5,
            patience=4, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            best_path, monitor='val_auc', mode='max',
            save_best_only=True, verbose=1),
        tf.keras.callbacks.TensorBoard(
            log_dir=tb_log, histogram_freq=0, update_freq='epoch'),
    ]
    cnn_srm.fit(
        train_srm_ds, validation_data=val_srm_ds,
        epochs=80, class_weight=class_weights,
        callbacks=callbacks, verbose=1
    )

    # ── STEP C: evaluate on ALL test targets ─────────────────────────────
    print(f'\n\nEVALUATION — CNN-SRM trained on {train_target}')
    best_model = tf.keras.models.load_model(best_path, compile=False)
    all_results[train_target] = {}

    for test_target in TEST_TARGETS:
        print(f'\n  Test target: {test_target}...')
        test_paths, test_lbls = load_split(test_target, 'test')
        test_srm_ds           = create_srm_dataset(
            test_paths, test_lbls, batch_size=BATCH_SIZE, shuffle=False)
        metrics, report       = evaluate_model(best_model, test_srm_ds)
        all_results[train_target][test_target] = metrics
        log_eval_to_tb(train_target, test_target, metrics)
        print_eval_report(train_target, test_target, metrics, report)

    # ── memory cleanup ────────────────────────────────────────────────────
    del autoencoder, encoder, cnn_srm, best_model, train_srm_ds, val_srm_ds
    gc.collect(); tf.keras.backend.clear_session()
    print(f'\n{train_target} experiment complete — GPU memory cleared.')

results_path = os.path.join(MODEL_DIR, 'all_results.pkl')
with open(results_path, 'wb') as f:
    pickle.dump(all_results, f)

print(f'\n\n{"="*70}')
print('  ALL EXPERIMENTS COMPLETE (5×5 matrix)')
print(f'{"="*70}')
print(f'\nResults saved → {results_path}')

In [ ]:
# =================== CELL 11: CROSS-DATABASE RESULTS MATRIX (5×5) ===================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

axis_names = list(DATABASES.keys()) + ['ALL']

for metric_key, metric_label, cmap in [
    ('roc_auc',  'ROC-AUC',  'YlOrRd'),
    ('accuracy', 'Accuracy', 'YlGn'),
    ('f1_fake',  'F1-FAKE',  'Blues'),
]:
    available = [d for d in axis_names if d in all_results]
    matrix = [
        [all_results[tr].get(te, {}).get(metric_key, float('nan')) for te in axis_names]
        for tr in available
    ]
    df = pd.DataFrame(
        matrix,
        index=[f'Train: {d}' for d in available],
        columns=[f'Test: {d}' for d in axis_names]
    )

    fig, ax = plt.subplots(figsize=(11, 7))
    sns.heatmap(
        df.astype(float), annot=True, fmt='.3f',
        cmap=cmap, vmin=0.40, vmax=1.00, ax=ax,
        linewidths=0.5, linecolor='gray',
        cbar_kws={'label': metric_label}
    )
    ax.set_title(
        f'CNN-SRM — Cross-Database {metric_label} (5×5)\n'
        '(diagonal = within-DB · last row/col = ALL combined)',
        fontsize=13, fontweight='bold'
    )
    ax.set_ylabel('Training Database', fontsize=11)
    ax.set_xlabel('Test Database',     fontsize=11)
    plt.xticks(rotation=30, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    fig_path = os.path.join(MODEL_DIR, f'cross_db_{metric_key}.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {fig_path}')
    print(f'\n{metric_label} Matrix:')
    print(df.to_string())
    print()

In [ ]:
# =================== CELL 12: SAVE RESULTS TABLE ===================

import pandas as pd

rows = []
for train_db, test_map in all_results.items():
    for test_db, metrics in test_map.items():
        rows.append({
            'model':    'cnn_srm',
            'train_db': train_db,
            'test_db':  test_db,
            **metrics,
        })

results_df = pd.DataFrame(rows)
csv_path   = os.path.join(MODEL_DIR, 'cross_db_results.csv')
results_df.to_csv(csv_path, index=False)

print('Results saved to:', csv_path)
print(f'Total rows: {len(results_df)}  (expected 25 for full 5×5)')
display(results_df)

In [ ]:
# =================== CELL 13: TENSORBOARD LAUNCH INSTRUCTIONS ===================

print('-' * 60)
print('  TENSORBOARD DASHBOARD')
print('-' * 60)
print()
print('1. On the SERVER terminal:')
print(f'   tensorboard --logdir {TB_LOG_ROOT} --port 6006 --bind_all')
print()
print('2. On your LOCAL machine, open an SSH tunnel:')
print('   ssh -L 6006:localhost:6006 <your_user>@<server_ip>')
print()
print('3. Open in browser:  http://localhost:6006')
print()
print('TensorBoard log structure for CNN-SRM (now includes ALL run):')
print(f'  {TB_LOG_ROOT}/cnn_srm/')
print('  ├── autoencoder_OpenForensics/   ← AE reconstruction loss')
print('  ├── autoencoder_CustomWar/')
print('  ├── autoencoder_CelebDF/')
print('  ├── autoencoder_CiFake/')
print('  ├── autoencoder_ALL/')
print('  ├── train_OpenForensics/         ← classifier accuracy/AUC')
print('  ├── train_CustomWar/')
print('  ├── train_CelebDF/')
print('  ├── train_CiFake/')
print('  ├── train_ALL/')
print('  └── cross_db_eval/')
print('      ├── train_OpenForensics__test_CustomWar/')
print('      ├── train_OpenForensics__test_ALL/')
print('      ├── train_ALL__test_OpenForensics/')
print('      └── ... (25 train→test pairs)')